In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from mord import LogisticAT


# =========================
# 1. LOAD DATA
# =========================

file_path = "Post Insights Sheet(Data).csv"
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)


# =========================
# 2. CREATE POSITION GROUPS
# =========================

position_map = {
    "QB": "Skill", "RB": "Skill", "FB": "Skill", "WR": "Skill", "TE": "Skill",
    "OT": "OL", "OG": "OL", "G": "OL", "C": "OL", "OC": "OL", "IOL": "OL", "OL": "OL",
    "EDGE": "DL", "DE": "DL", "DT": "DL", "DL": "DL",
    "LB": "LB", "ILB": "LB", "OLB": "LB",
    "CB": "Secondary", "S": "Secondary", "SAF": "Secondary", "DB": "Secondary",
    "K": "Special Teams", "P": "Special Teams", "LS": "Special Teams"
}

df["Position Group"] = df["Position"].map(position_map).fillna(df["Position"])


# =========================
# 3. CREATE ORDINAL DRAFT TIER TARGET
# =========================

def draft_tier(pick):
    if pick <= 32:
        return "Round 1"
    elif pick <= 96:
        return "Rounds 2-3"
    elif pick <= 160:
        return "Rounds 4-5"
    elif pick <= 262:
        return "Rounds 6-7"
    else:
        return "Undrafted"


df["Draft Tier"] = df["Overall Pick"].apply(draft_tier)

labels_order = [
    "Round 1",
    "Rounds 2-3",
    "Rounds 4-5",
    "Rounds 6-7",
    "Undrafted"
]

tier_to_num = {
    "Round 1": 0,
    "Rounds 2-3": 1,
    "Rounds 4-5": 2,
    "Rounds 6-7": 3,
    "Undrafted": 4
}

num_to_tier = {
    0: "Round 1",
    1: "Rounds 2-3",
    2: "Rounds 4-5",
    3: "Rounds 6-7",
    4: "Undrafted"
}

df["Draft Tier Ordinal"] = df["Draft Tier"].map(tier_to_num)

print("\nDraft Tier Counts:")
print(df["Draft Tier"].value_counts())


# =========================
# 4. FEATURES
# ELO, Arm (inches), and School removed
# =========================

features = [
    "Season",
    "Position",
    "Position Group",
    "Conference",
    "Height (inches)",
    "Weight(lbs)",
    "Hand (inches)",
    "Wingspan (inches)",
    "40-Yard Dash (seconds)",
    "Vertical (inches)",
    "Bench (reps)",
    "Broad Jump (inches)",
    "3-Cone (seconds)",
    "Shuttle (seconds)",
    "Combine Participation",
    "FPI",
    "Efficiencies Overall",
    "Efficiencies Offense",
    "Efficiencies Defense",
    "Efficiencies SpecialTeams",
    "CFP Appearance",
    "National Award"
]

combine_metrics = [
    "Height (inches)",
    "Weight(lbs)",
    "Hand (inches)",
    "Wingspan (inches)",
    "40-Yard Dash (seconds)",
    "Vertical (inches)",
    "Bench (reps)",
    "Broad Jump (inches)",
    "3-Cone (seconds)",
    "Shuttle (seconds)"
]

basic_features = [
    "Season",
    "Position",
    "Conference",
    "CFP Appearance"
]


# =========================
# 5. BUILD ORDINAL MODEL FUNCTION
# =========================

def build_model(feature_list):
    categorical_features = [
        col for col in ["Position", "Position Group", "Conference"]
        if col in feature_list
    ]

    numeric_features = [
        col for col in feature_list if col not in categorical_features
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            ("num", "passthrough", numeric_features)
        ]
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("ordinal_model", LogisticAT(alpha=1.0))
        ]
    )

    return model


# =========================
# 6. MANUAL ORDINAL PREDICTION REPORT
# =========================

def print_manual_report(y_true, y_pred, title):
    y_true_labels = pd.Series(y_true).map(num_to_tier)
    y_pred_labels = pd.Series(y_pred).map(num_to_tier)

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true_labels,
        y_pred_labels,
        labels=labels_order,
        zero_division=0
    )

    print(f"\n{title}")
    print(f"{'Tier':<14}{'Precision':>12}{'Recall':>12}{'F1-Score':>12}{'Support':>12}")

    for label, p, r, f, s in zip(labels_order, precision, recall, f1, support):
        print(f"{label:<14}{p:>12.2f}{r:>12.2f}{f:>12.2f}{s:>12}")

    print("\nAccuracy:", round(accuracy_score(y_true, y_pred), 4))

    ordinal_error = np.mean(np.abs(np.array(y_true) - np.array(y_pred)))
    print("Average Ordinal Error:", round(ordinal_error, 4))


# =========================
# 7. OVERALL ORDINAL MODEL
# =========================

X = df[features]
y = df["Draft Tier Ordinal"]

overall_model = build_model(features)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    overall_model,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("\nOVERALL ORDINAL MODEL - 5-Fold CV Accuracy Scores:")
print(cv_scores)
print("Average CV Accuracy:", round(cv_scores.mean(), 4))
print("CV Accuracy Standard Deviation:", round(cv_scores.std(), 4))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

overall_model.fit(X_train, y_train)
y_pred = overall_model.predict(X_test)

print_manual_report(
    y_test,
    y_pred,
    "OVERALL MODEL - Ordinal Prediction Report"
)


# =========================
# 8. BASIC MODEL FOR OVERALL GAINS CHARTS
# =========================

X_basic_full = df[basic_features]

X_train_basic = X_basic_full.loc[X_train.index]
X_test_basic = X_basic_full.loc[X_test.index]

overall_basic_model = build_model(basic_features)
overall_basic_model.fit(X_train_basic, y_train)

basic_probs_overall = overall_basic_model.predict_proba(X_test_basic)

basic_prob_df_overall = pd.DataFrame(
    basic_probs_overall,
    columns=labels_order
)


# =========================
# 9. OVERALL FEATURE COEFFICIENTS
# =========================

ordinal_model = overall_model.named_steps["ordinal_model"]
feature_names = overall_model.named_steps["preprocessor"].get_feature_names_out()

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": ordinal_model.coef_
}).sort_values(by="Coefficient", key=abs, ascending=False)

print("\nTop 20 Overall Features by Absolute Coefficient Size:")
print(importance_df.head(20).to_string(index=False))


# =========================
# 10. POSITION GROUP ORDINAL MODELS
# =========================

position_group_results = []
position_group_importances = []
group_model_store = {}

for group in df["Position Group"].dropna().unique():
    group_df = df[df["Position Group"] == group].copy()

    print("\n" + "=" * 70)
    print(f"POSITION GROUP ORDINAL MODEL: {group}")
    print("=" * 70)
    print("Rows:", len(group_df))
    print("Draft Tier Counts:")
    print(group_df["Draft Tier"].value_counts())

    if len(group_df) < 50 or group_df["Draft Tier Ordinal"].nunique() < 2:
        print("Skipped: not enough data or only one draft tier present.")
        continue

    group_features = [
        col for col in features if col != "Position Group"
    ]

    X_group = group_df[group_features]
    y_group = group_df["Draft Tier Ordinal"]

    min_class_count = y_group.value_counts().min()

    if min_class_count < 2:
        print("Skipped: at least one tier has fewer than 2 players.")
        continue

    n_splits = min(5, min_class_count)

    group_model = build_model(group_features)

    group_cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    group_cv_scores = cross_val_score(
        group_model,
        X_group,
        y_group,
        cv=group_cv,
        scoring="accuracy"
    )

    print(f"\n{n_splits}-Fold Stratified CV Accuracy Scores:")
    print(group_cv_scores)
    print("Average CV Accuracy:", round(group_cv_scores.mean(), 4))
    print("CV Accuracy Standard Deviation:", round(group_cv_scores.std(), 4))

    X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
        X_group,
        y_group,
        test_size=0.25,
        random_state=42,
        stratify=y_group
    )

    group_model.fit(X_train_g, y_train_g)
    y_pred_g = group_model.predict(X_test_g)

    print_manual_report(
        y_test_g,
        y_pred_g,
        f"{group} - Ordinal Prediction Report"
    )

    position_group_results.append({
        "Position Group": group,
        "Rows": len(group_df),
        "CV Accuracy Mean": group_cv_scores.mean(),
        "CV Accuracy Std": group_cv_scores.std(),
        "Test Accuracy": accuracy_score(y_test_g, y_pred_g),
        "Average Ordinal Error": np.mean(np.abs(np.array(y_test_g) - np.array(y_pred_g)))
    })

    ordinal_model_g = group_model.named_steps["ordinal_model"]
    feature_names_g = group_model.named_steps["preprocessor"].get_feature_names_out()

    importance_g = pd.DataFrame({
        "Feature": feature_names_g,
        "Coefficient": ordinal_model_g.coef_
    }).sort_values(by="Coefficient", key=abs, ascending=False)

    print(f"\nTop 15 Features for {group} by Absolute Coefficient Size:")
    print(importance_g.head(15).to_string(index=False))

    combine_group_importance = []

    for metric in combine_metrics:
        matching_rows = importance_g[
            importance_g["Feature"].str.contains(metric, regex=False)
        ]

        combine_group_importance.append({
            "Position Group": group,
            "Metric": metric,
            "Coefficient Strength": matching_rows["Coefficient"].abs().sum()
        })

    combine_group_df = pd.DataFrame(combine_group_importance).sort_values(
        by="Coefficient Strength",
        ascending=False
    )

    print(f"\nCombine Metric Coefficient Strength for {group}:")
    print(combine_group_df.to_string(index=False))

    position_group_importances.append(combine_group_df)

    group_basic_features = [
        col for col in basic_features
        if col in group_df.columns
    ]

    X_group_basic = group_df[group_basic_features]
    X_train_gb = X_group_basic.loc[X_train_g.index]
    X_test_gb = X_group_basic.loc[X_test_g.index]

    group_basic_model = build_model(group_basic_features)
    group_basic_model.fit(X_train_gb, y_train_g)

    basic_probs_g = group_basic_model.predict_proba(X_test_gb)
    basic_prob_df_g = pd.DataFrame(
        basic_probs_g,
        columns=labels_order
    )

    full_probs_g = group_model.predict_proba(X_test_g)

    group_model_store[group] = {
        "full_prob_df": pd.DataFrame(full_probs_g, columns=labels_order),
        "basic_prob_df": basic_prob_df_g,
        "y_test": y_test_g,
        "X_test": X_test_g
    }


# =========================
# 11. SAVE POSITION GROUP SUMMARY
# =========================

position_group_results_df = pd.DataFrame(position_group_results)

print("\nPOSITION GROUP ORDINAL MODEL SUMMARY:")
print(position_group_results_df.to_string(index=False))

position_group_results_df.to_csv(
    "position_group_ordinal_model_summary_no_ELO_no_arm_no_school.csv",
    index=False
)

if position_group_importances:
    all_position_importances_df = pd.concat(
        position_group_importances,
        ignore_index=True
    )

    all_position_importances_df.to_csv(
        "position_group_combine_metric_coefficient_strength_no_ELO_no_arm_no_school.csv",
        index=False
    )


# =========================
# 12. GAINS CHART FUNCTION
# =========================

def plot_gains_chart(
    target_tier,
    gains_base_arg,
    prob_df_arg,
    basic_prob_df_arg=None,
    label_prefix=""
):
    temp_df = gains_base_arg.copy()

    temp_df["Actual Target"] = np.where(
        temp_df["Actual Draft Tier"] == target_tier,
        1,
        0
    )

    temp_df["Predicted Probability"] = prob_df_arg[target_tier].values

    temp_df = temp_df.sort_values(
        by="Predicted Probability",
        ascending=False
    ).reset_index(drop=True)

    total_actual_targets = temp_df["Actual Target"].sum()

    if total_actual_targets == 0:
        print(f"No actual {target_tier} players in this test set.")
        return None

    temp_df["Cumulative Actual Captured"] = temp_df["Actual Target"].cumsum()

    temp_df["Percent of Players Reviewed"] = (
        np.arange(1, len(temp_df) + 1) / len(temp_df)
    ) * 100

    temp_df["Percent of Actual Tier Captured"] = (
        temp_df["Cumulative Actual Captured"] / total_actual_targets
    ) * 100

    plt.figure(figsize=(10, 6))

    plt.plot(
        temp_df["Percent of Players Reviewed"],
        temp_df["Percent of Actual Tier Captured"],
        label=f"Ordinal Model: {target_tier}"
    )

    if basic_prob_df_arg is not None:
        actual_raw = (
            gains_base_arg["Actual Draft Tier"] == target_tier
        ).astype(int).values

        basic_col = basic_prob_df_arg[target_tier].values
        order_basic = np.argsort(-basic_col)

        cum_basic = (
            actual_raw[order_basic].cumsum()
            / total_actual_targets
            * 100
        )

        pct_x = (
            np.arange(1, len(actual_raw) + 1)
            / len(actual_raw)
            * 100
        )

        plt.plot(
            pct_x,
            cum_basic,
            linestyle="-.",
            label="Basic Model"
        )

    plt.plot(
        temp_df["Percent of Players Reviewed"],
        temp_df["Percent of Players Reviewed"],
        linestyle="--",
        label="Random Baseline"
    )

    title_prefix = f"{label_prefix}: " if label_prefix else ""

    plt.title(f"{title_prefix}Gains Chart: Predicting {target_tier} Players")
    plt.xlabel("Percent of Players Reviewed")
    plt.ylabel(f"Percent of Actual {target_tier} Players Captured")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return temp_df


# =========================
# 13. OVERALL GAINS CHARTS
# =========================

predicted_probs = overall_model.predict_proba(X_test)

prob_df = pd.DataFrame(
    predicted_probs,
    columns=labels_order
)

gains_base = X_test.copy().reset_index(drop=True)
gains_base["Actual Draft Tier"] = y_test.reset_index(drop=True).map(num_to_tier)

tier_outputs = {}

for tier in labels_order:
    print(f"\nCreating overall gains chart for: {tier}")

    tier_outputs[tier] = plot_gains_chart(
        target_tier=tier,
        gains_base_arg=gains_base,
        prob_df_arg=prob_df,
        basic_prob_df_arg=basic_prob_df_overall
    )

for tier, output_df in tier_outputs.items():
    if output_df is not None:
        clean_name = (
            tier.replace(" ", "_")
                .replace("-", "_")
                .replace("/", "_")
        )

        output_df.to_csv(
            f"gains_chart_{clean_name}_ordinal_no_ELO_no_arm_no_school.csv",
            index=False
        )


# =========================
# 14. POSITION GROUP GAINS CHARTS
# =========================

for group, store in group_model_store.items():
    print(f"\n--- Gains Charts: {group} ---")

    y_test_g_labels = pd.Series(store["y_test"]).reset_index(drop=True).map(num_to_tier)

    group_gains_base = store["X_test"].copy().reset_index(drop=True)
    group_gains_base["Actual Draft Tier"] = y_test_g_labels.values

    group_prob_df = store["full_prob_df"].reset_index(drop=True)
    group_basic_prob_df = store["basic_prob_df"].reset_index(drop=True)

    for tier in labels_order:
        print(f"Creating {group} gains chart for: {tier}")

        plot_gains_chart(
            target_tier=tier,
            gains_base_arg=group_gains_base,
            prob_df_arg=group_prob_df,
            basic_prob_df_arg=group_basic_prob_df,
            label_prefix=group
        )


# =========================
# 15. CHECK NUMBER OF COEFFICIENTS AND THRESHOLDS
# =========================

ordinal_model = overall_model.named_steps["ordinal_model"]

print("\nNumber of coefficients:", len(ordinal_model.coef_))
print("Number of thresholds:", len(ordinal_model.theta_))

print("\nDone. Ordinal models, coefficient tables, basic model, and gains charts saved.")

Dataset shape: (965, 27)

Draft Tier Counts:
Draft Tier
Undrafted     264
Rounds 6-7    230
Rounds 2-3    189
Rounds 4-5    186
Round 1        96
Name: count, dtype: int64
